# Evaluación Parcial 1 - Machine Learning

**Asignatura:** MLY0100 Machine Learning  
**Integrantes:** Christian Sandoval, Nicolás Vega  
**Dataset:** DS1-18-Datos-Properati.csv  
**Fecha:** 25 de septiembre de 2026

# Fase 1 - Comprensión del Negocio

## Contexto y objetivo

El dataset contiene publicaciones de propiedades en venta en Argentina. El objetivo es entender sus principales características y preparar los datos antes de trabajar con modelos de Machine Learning.

**Pregunta analítica:** ¿Qué características de las propiedades, como ubicación, superficie y tipo de propiedad, se relacionan con las diferencias de precio?

**Supuestos:** los valores faltantes no se consideran cero, los valores extremos se revisan antes de modificarlos y las conclusiones corresponden al dataset entregado.

## Targets

- **Regresión:** `price`, porque es una variable numérica continua.
- **Clasificación:** `property_type`, porque contiene categorías como Departamento, Casa y PH.

En esta evaluación no se entrenan modelos. Solo se identifican los posibles targets.


# Fase 2 - Comprensión de los Datos

## Carga de librerías y datos

Se importan las librerías que se usarán y se carga el CSV que viene dentro del ZIP entregado.


In [ ]:
%matplotlib inline

from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

ruta_zip = Path('Evaluación Parcial 1 - Dataset.zip')

if not ruta_zip.exists():
    ruta_zip = Path('../Evaluación Parcial 1 - Dataset.zip')

if not ruta_zip.exists():
    from google.colab import files
    uploaded = files.upload()
    ruta_zip = Path(next(iter(uploaded.keys())))

with zipfile.ZipFile(ruta_zip, 'r') as archivo_zip:
    nombre_csv = [
        nombre for nombre in archivo_zip.namelist()
        if nombre.endswith('DS1-18-Datos-Properati.csv')
        and not nombre.startswith('__MACOSX')
    ][0]

    with archivo_zip.open(nombre_csv) as archivo_csv:
        df = pd.read_csv(archivo_csv)

print('Dimensiones:', df.shape)
df.head()


## Inspección de la estructura

Se revisan columnas, tipos de datos, valores no nulos y estadísticas generales.


In [ ]:
print(df.dtypes)
df.info()
df.describe(include='all').T


El dataset tiene **146.660 filas y 19 columnas**. Las fechas están inicialmente como texto y se observan valores faltantes en `lat`, `lon`, `bathrooms`, `surface_total` y `surface_covered`.


## Estadísticos descriptivos

Se calculan media, mediana, moda, desviación estándar, varianza e IQR para las variables numéricas principales.


In [ ]:
variables_analisis = [
    'rooms', 'bedrooms', 'bathrooms',
    'surface_total', 'surface_covered', 'price'
]

estadisticos = pd.DataFrame({
    'media': df[variables_analisis].mean(),
    'mediana': df[variables_analisis].median(),
    'moda': df[variables_analisis].mode().iloc[0],
    'desviacion_estandar': df[variables_analisis].std(),
    'varianza': df[variables_analisis].var(),
    'Q1': df[variables_analisis].quantile(0.25),
    'Q3': df[variables_analisis].quantile(0.75)
})

estadisticos['IQR'] = estadisticos['Q3'] - estadisticos['Q1']
estadisticos.round(2)


`price`, `surface_total` y `surface_covered` muestran diferencias importantes entre media y mediana. Esto indica asimetría y la posible presencia de valores extremos.


## Distribuciones y visualizaciones

Se revisan las distribuciones con histograma, boxplots, barras, scatter y heatmap. El percentil 99 se usa solamente para que algunos gráficos sean legibles, sin modificar todavía los datos.


In [ ]:
limite_price = df['price'].quantile(0.99)
limite_superficie = df['surface_total'].quantile(0.99)

plt.figure(figsize=(8, 5))
plt.hist(df.loc[df['price'] <= limite_price, 'price'].dropna(), bins=40, edgecolor='black')
plt.title('Distribución del precio')
plt.xlabel('Precio (USD)')
plt.ylabel('Cantidad de propiedades')
plt.show()

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].boxplot(df.loc[df['price'] <= limite_price, 'price'].dropna(), vert=False)
axs[0].set_title('Boxplot de price')
axs[0].set_xlabel('Precio (USD)')
axs[0].set_ylabel('Distribución')
axs[1].boxplot(df.loc[df['surface_total'] <= limite_superficie, 'surface_total'].dropna(), vert=False)
axs[1].set_title('Boxplot de surface_total')
axs[1].set_xlabel('Superficie total (m²)')
axs[1].set_ylabel('Distribución')
plt.tight_layout()
plt.show()

tipos_propiedad = df['property_type'].value_counts().head(10)
plt.figure(figsize=(9, 5))
plt.barhhtipos_propiedad.index, tipos_propiedad.values)
plt.title('10 tipos de propiedad con más publicaciones')
plt.xlabel('Cantidad de publicaciones')
plt.ylabel('Tipo de propiedad')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
datos_scatter = df[
    (df['surface_total'] <= limite_superficie) &
    (df['price'] <= limite_price)
][['surface_total', 'price']].dropna()

muestra_scatter = datos_scatter.sample(n=min(5000, len(datos_scatter)), random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(muestra_scatter['surface_total'], muestra_scatter['price'], alpha=0.35)
plt.title('Relación entre superficie total y precio')
plt.xlabel('Superficie total (m²)')
plt.ylabel('Precio (USD)')
plt.show()

matriz_correlacion = df[variables_analisis].corr()
plt.figure(figsize=(9, 6))
sns.heatmap(matriz_correlacion, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlación')
plt.xlabel('Variables')
plt.ylabel('Variables')
plt.show()


Los gráficos muestran una distribución de precios asimétrica y una tendencia positiva entre superficie total y precio, aunque con bastante dispersión. `Departamento` es la clase más frecuente. La correlación más alta entre las variables revisadas aparece entre `rooms` y `bedrooms`; para `price`, una de las relaciones lineales más altas aparece con `bathrooms`.


# Fase 3 - Preparación de los Datos

## Missing values

Primero se revisa la cantidad y porcentaje de datos faltantes. También se compara la ausencia según `property_type` para aproximarnos al mecanismo de missing values.


In [ ]:
resumen_nulos = pd.DataFrame({
    'cantidad_nulos': df.isna().sum(),
    'porcentaje_nulos': (df.isna().mean() * 100).round(2)
})
resumen_nulos = resumen_nulos[resumen_nulos['cantidad_nulos'] > 0]
resumen_nulos = resumen_nulos.sort_values('porcentaje_nulos', ascending=False)
display(resumen_nulos)

plt.figure(figsize=(8, 5))
plt.barh(resumen_nulos.index, resumen_nulos['porcentaje_nulos'])
plt.title('Porcentaje de valores faltantes')
plt.xlabel('Porcentaje (%)')
plt.ylabel('Variable')
plt.gca().invert_yaxis()
plt.show()

for columna in ['surface_total', 'surface_covered', 'bathrooms']:
    porcentaje_por_tipo = (
        df.assign(faltante=df[columna].isna())
          .groupby('property_type', observed=False)['faltante']
          .mean()
          .mul(100)
          .sort_values(ascending=False)
          .round(2)
    )
    print(f'\nNulos en {columna} según property_type:')
    print(porcentaje_por_tipo.head(10))

print('\nRegistros con lat y lon faltantes al mismo tiempo:',
      (df['lat'].isna() & df['lon'].isna()).sum())


Los porcentajes de nulos cambian según `property_type` y `lat` y `lon` faltan juntos en muchos registros. Para este trabajo se consideran principalmente compatibles con **MAR (Missing At Random)**, porque la ausencia parece depender de variables observadas. Es una interpretación de trabajo, no una prueba definitiva.


### Prueba con KNNImputer

En clases se trabajó `KNNImputer` con variables numéricas. Se prueba en una muestra de 1.500 filas para evitar un cálculo demasiado lento con las 146 mil filas.


In [ ]:
variables_knn = ['bathrooms', 'surface_total', 'surface_covered']
muestra_knn = df[variables_knn].sample(n=1500, random_state=42).copy()

knn_imputer = KNNImputer(n_neighbors=2, weights='uniform')

datos_knn = muestra_knn.copy()
datos_knn[variables_knn] = knn_imputer.fit_transform(datos_knn[variables_knn])

pd.DataFrame({
    'media_original': muestra_knn.mean(),
    'media_knn': datos_knn.mean(),
    'std_original': muestra_knn.std(),
    'std_knn': datos_knn.std()
}).round(2)


KNN completa los nulos usando observaciones cercanas. Para el tratamiento final se usa la **mediana agrupada**, porque es simple de explicar y es menos sensible a valores extremos que la media.


In [ ]:
df_limpio = df.copy()

for columna in ['surface_total', 'surface_covered', 'bathrooms']:
    mediana = df_limpio.groupby(
        'property_type', observed=False
    )[columna].transform('median')
    df_limpio[columna] = df_limpio[columna].fillna(mediana)

for columna in ['lat', 'lon']:
    mediana = df_limpio.groupby(
        'l3', observed=False
    )[columna].transform('median')
    df_limpio[columna] = df_limpio[columna].fillna(mediana)

print('Valores nulos restantes:', int(df_limpio.isna().sum().sum()))
print('Dimensiones:', df_limpio.shape)


Después de la imputación quedan **0 valores nulos** y se mantienen las **146.660 filas**.


## Outliers

Los valores atípicos se detectan con **IQR** y **Z-score**. Primero se comparan ambos resultados y después se decide el tratamiento.


In [ ]:
resumen_outliers = []

for columna in variables_analisis:
    q1 = df_limpio[columna].quantile(0.25)
    q3 = df_limpio[columna].quantile(0.75)
    iqr = q3 - q1
    inferior = q1 - 1.5 * iqr
    superior = q3 + 1.5 * iqr

    mascara_iqr = (
        (df_limpio[columna] < inferior) |
        (df_limpio[columna] > superior)
    )

    z_scores = np.abs(stats.zscore(df_limpio[columna]))

    resumen_outliers.append({
        'variable': columna,
        'outliers_iqr': int(mascara_iqr.sum()),
        'porcentaje_iqr': round(mascara_iqr.mean() * 100, 2),
        'outliers_zscore': int((z_scores > 3).sum()),
        'porcentaje_zscore': round((z_scores > 3).mean() * 100, 2)
    })

resumen_outliers = pd.DataFrame(resumen_outliers)
display(resumen_outliers)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].boxplot(df_limpio['price'], vert=False)
axs[0].set_title('Boxplot de price')
axs[0].set_xlabel('Precio (USD)')
axs[0].set_ylabel('Distribución')
axs[1].boxplot(df_limpio['surface_total'], vert=False)
axs[1].set_title('Boxplot de surface_total')
axs[1].set_xlabel('Superficie total (m²)')
axs[1].set_ylabel('Distribución')
plt.tight_layout()
plt.show()


IQR detecta más casos que Z-score, especialmente en precio y superficies. En propiedades un valor alto no necesariamente es un error, por lo que no se eliminarán automáticamente todas las filas detectadas.


### Tratamiento de outliers

Se usa el **percentil 99** como límite superior. Los valores que lo superan se reemplazan por ese límite. Así se controlan los casos más extremos sin eliminar filas.


In [ ]:
variables_outliers = [
    'rooms', 'bedrooms', 'bathrooms',
    'surface_total', 'surface_covered', 'price'
]

resumen_tratamiento = []

for columna in variables_outliers:
    limite_p99 = df_limpio[columna].quantile(0.99)
    maximo_antes = df_limpio[columna].max()
    cantidad = int((df_limpio[columna] > limite_p99).sum())

    df_limpio[columna] = df_limpio[columna].clip(upper=limite_p99)

    resumen_tratamiento.append({
        'variable': columna,
        'limite_p99': limite_p99,
        'valores_ajustados': cantidad,
        'maximo_antes': maximo_antes,
        'maximo_despues': df_limpio[columna].max()
    })

pd.DataFrame(resumen_tratamiento).round(2)


Después del tratamiento se conservan todas las filas. Solo se reduce el efecto de los valores superiores más extremos.


## Estandarización

En clases se trabajó `StandardScaler`. Como los valores extremos ya fueron controlados, se aplica a las variables numéricas principales. Las columnas originales se mantienen para seguir interpretando los datos en sus unidades normales.


In [ ]:
print('Asimetría después del tratamiento de outliers:')
print(df_limpio[variables_analisis].skew().round(3))

scaler = StandardScaler()
array_scaler = scaler.fit_transform(df_limpio[variables_analisis])

df_scaler = pd.DataFrame(
    array_scaler,
    columns=variables_analisis,
    index=df_limpio.index
)

df_escalado = df_limpio.copy()
columnas_scaler = []

for columna in variables_analisis:
    nueva = columna + '_std'
    df_escalado[nueva] = df_scaler[columna]
    columnas_scaler.append(nueva)

print('\nAntes:')
display(df_limpio[variables_analisis].agg(['mean', 'std', 'min', 'max']).T.round(2))

print('Después de StandardScaler:')
display(df_escalado[columnas_scaler].agg(['mean', 'std', 'min', 'max']).T.round(2))


Las nuevas columnas estandarizadas quedan con media cercana a **0** y desviación estándar cercana a **1**. Se conservan también las variables originales.


## Dataset final

Se corrigen los tipos de datos y se revisa el resultado final.


In [ ]:
df_final = df_escalado.copy()

for columna in ['start_date', 'end_date', 'created_on']:
    df_final[columna] = df_final[columna].astype('datetime64[s]')

for columna in ['l1', 'l2', 'l3', 'currency', 'property_type', 'operation_type']:
    df_final[columna] = df_final[columna].astype('category')

print('Dimensiones del dataset final:', df_final.shape)
print('Valores nullos:', int(df_final.isna().sum().sum()))
print(df_final.dtypes)


El dataset final mantiene **146.660 filas**, queda con **25 columnas** y no tiene valores nulos.


### Exportación

Se guarda el dataset preparado en un nuevo CSV.


In [ ]:
df_final.to_csv('DS1-18-Datos-Properati-preparado.csv', index=False)
print('Dataset exportado correctamente.')


# Conclusiones

Se completaron las tres primeras fases de CRISP-DM: **Comprensión del Negocio, Comprensión de los Datos y Preparación de los Datos**.

Los gráficos muestran que la superficie total tiene una tendencia positiva con el precio, aunque existe bastante dispersión. También aparecen relaciones entre el precio y variables como baños, ambientes y dormitorios, por lo que el precio no depende de una sola característica.

Se encontraron missing values en superficies, coordenadas y baños. Se comparó KNNImputer con una estrategia de medianas agrupadas y finalmente se usaron medianas para completar los datos sin eliminar filas.

Los outliers se revisaron con IQR y Z-score. Como un valor extremo puede representar una propiedad real, solo se controlaron los casos superiores más extremos con el percentil 99.

Finalmente se aplicó `StandardScaler` y se corrigieron los tipos de datos. El dataset queda preparado para continuar posteriormente con modelado.
